# 00 — Data loading & schema validation

**Purpose:** confirm the meta-learning `.csv.gz` files load correctly, and dump a
compact schema report for two representative tasks (one classification, one
regression) across all four output kinds — plus a schema-consistency check across
all 31 tasks.

This notebook is **self-contained**: it does *not* import `utils.py`. Once it runs
clean, we fold the validated logic into `utils.py`.

Run all cells, then either paste the output back to me, or just upload the
generated **`schema_report.txt`** (written next to this notebook).

> Note: the inner files (`in_splits`, `in_agg`) are large and may take a little
> while to load — that's expected. The output stays compact regardless (heads are
> truncated and only low-cardinality columns get value counts).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# --- report helper: print AND collect, so you can paste or upload the file ---
report_lines = []
def log(*args):
    s = " ".join(str(a) for a in args)
    print(s)
    report_lines.append(s)

# --- resolve the data folder relative to this notebook (no personal paths) ---
def find_data_dir():
    here = Path.cwd()
    for root in [here, *here.parents]:
        for cand in [root / "data" / "metalearning_output_datasets",
                     root / "metalearning_output_datasets"]:
            if cand.exists():
                return cand
    raise FileNotFoundError("Could not find metalearning_output_datasets/ near this notebook.")

DATA_DIR = find_data_dir()
KINDS = ("in_agg", "in_splits", "out_agg", "out_splits")
log("pandas", pd.__version__, "| numpy", np.__version__)
log("DATA_DIR:", DATA_DIR)

In [ ]:
# --- discover tasks and confirm each has all four kinds ---
import collections
files = sorted(DATA_DIR.glob("*.csv.gz"))
log(f"{len(files)} .csv.gz files found")

present = collections.defaultdict(set)
for f in files:
    stem = f.name[: -len(".csv.gz")]
    task, kind = stem.rsplit("__", 1)
    present[task].add(kind)

tasks = sorted(present)
log(f"{len(tasks)} tasks:")
for t in tasks:
    log("   ", t)

missing = {t: sorted(set(KINDS) - present[t]) for t in tasks if set(KINDS) - present[t]}
log("Tasks missing kinds:", missing if missing else "none - all tasks have all 4 kinds")

In [ ]:
# --- compact inspector: loads a file and reports its schema ---
def inspect(task, kind, cat_threshold=30):
    path = DATA_DIR / f"{task}__{kind}.csv.gz"          # .gz auto-detected
    df = pd.read_csv(path)
    log("=" * 72)
    log(f"{task}  __  {kind}")
    log(f"shape: {df.shape}   in-RAM: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
    log(f"index: {type(df.index).__name__}  names={df.index.names}")
    log("-- dtypes --")
    log(df.dtypes.to_string())
    log("-- head(3), transposed for readability --")
    log(df.head(3).T.to_string())
    log("-- columns --")
    for c in df.columns:
        nu = df[c].nunique(dropna=False)
        if nu <= cat_threshold:
            vc = df[c].value_counts(dropna=False).head(cat_threshold)
            log(f"[{c}] nunique={nu}: {dict(vc)}")
        else:
            log(f"[{c}] nunique={nu} (high-card; e.g. {list(df[c].dropna().unique()[:3])})")
    return df

CLS = "ext_qual_binary"        # classification example
REG = "ext_quant_avg_force"    # regression example

dfs = {}
for task in (CLS, REG):
    for kind in KINDS:
        dfs[(task, kind)] = inspect(task, kind)

In [ ]:
# --- cheap cross-task consistency check: read headers only (nrows=0) ---
from collections import Counter
layouts = {}
for t in tasks:
    for kind in KINDS:
        p = DATA_DIR / f"{t}__{kind}.csv.gz"
        if p.exists():
            layouts[(t, kind)] = tuple(pd.read_csv(p, nrows=0).columns)

log("=" * 72)
log("Column-layout consistency across tasks (by kind):")
for kind in KINDS:
    sets = Counter(cols for (t, k), cols in layouts.items() if k == kind)
    log(f"  {kind}: {len(sets)} distinct layout(s) across {sum(sets.values())} tasks")
    common, n = sets.most_common(1)[0]
    log(f"     most common ({n} tasks), {len(common)} cols: {list(common)}")

# --- write the report so you can upload instead of pasting ---
out = Path.cwd() / "schema_report.txt"
out.write_text("\n".join(report_lines), encoding="utf-8")
print(f"\nWrote {out} ({out.stat().st_size/1024:.0f} KB) - upload this or paste the output above.")